In [38]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
import litellm
litellm.ssl_verify = False

In [2]:
import sys
sys.path.append(".")

In [3]:
from leftover import LeftoversCrew

In [4]:
from pydantic import BaseModel, Field
from typing import List, Dict, Optional
from IPython.display import display, Markdown, JSON
from datetime import datetime
import os

Creating a grocery shopping assistant structure

GroceryItem

In [5]:
class GroceryItem(BaseModel):
    name: str = Field(description="Name of the grocery item")
    quantity: str = Field(description="Quantity needed (for example, '2 lbs', '1 gallon')")
    estimated_price: str = Field(description="Estimated price (for example, '$3-5')")
    category: str = Field(description="Store section (for example, 'Produce', 'Dairy')")

In [6]:
sample_item = GroceryItem(
    name="Chicken Breast",
    quantity="2 lbs",
    estimated_price="$8-12",
    category="Meat"
)

In [7]:
sample_item

GroceryItem(name='Chicken Breast', quantity='2 lbs', estimated_price='$8-12', category='Meat')

In [8]:
type(sample_item)

__main__.GroceryItem

In [9]:
# Display structured data
print("🛒 Sample Grocery Item Structure:")
display(JSON(sample_item.model_dump()))

🛒 Sample Grocery Item Structure:


<IPython.core.display.JSON object>

In [11]:
class MealPlan(BaseModel):
    """Simple meal plan"""
    meal_name: str = Field(description="Name of the meal")
    difficulty_level: str = Field(description="'Easy', 'Medium', 'Hard'")
    servings: int = Field(description="Number of people it serves")
    researched_ingredients: list[str] = Field(description="Ingredients found through research")

In [12]:
sample_meal = MealPlan(
    meal_name="Chicken Stir Fry",
    difficulty_level="Easy",
    servings=4,
    researched_ingredients=["chicken breast", "broccoli", "bell peppers", "garlic", "soy sauce", "rice"]
)

In [13]:
sample_meal

MealPlan(meal_name='Chicken Stir Fry', difficulty_level='Easy', servings=4, researched_ingredients=['chicken breast', 'broccoli', 'bell peppers', 'garlic', 'soy sauce', 'rice'])

In [14]:
print("\n🍽️ Sample Meal Plan Structure:")
display(JSON(sample_meal.model_dump()))


🍽️ Sample Meal Plan Structure:


<IPython.core.display.JSON object>

ShoppingCategory

In [15]:
class ShoppingCategory(BaseModel):
    """Store section with items"""
    section_name: str = Field(description="Store section (for example, 'Produce', 'Dairy')")
    items: list[GroceryItem] = Field(description="Items in this section")
    estimated_total: str = Field(description="Estimated cost for this section")

In [16]:
sample_section = ShoppingCategory(
    section_name="Produce",
    items=[
        GroceryItem(name="Bell Peppers", quantity="3 pieces", estimated_price="$3-4", category="Produce"),
        GroceryItem(name="Onions", quantity="2 lbs", estimated_price="$2-3", category="Produce")
    ],
    estimated_total="$5-7"
)

In [17]:
print("\n🏪 Sample Shopping Section:")
display(JSON(sample_section.model_dump()))


🏪 Sample Shopping Section:


<IPython.core.display.JSON object>

GroceryShoppingPlan

In [18]:
class GroceryShoppingPlan(BaseModel):
    """Complete simplified shopping plan"""
    total_budget: str = Field(description="Total planned budget")
    meal_plans: list[MealPlan] = Field(description="Planned meals")
    shopping_sections: list[ShoppingCategory] = Field(description="Organized by store sections")
    shopping_tips: list[str] = Field(description="Money-saving and efficiency tips")

In [ ]:
# weekly_shopping_plan = GroceryShoppingPlan(
#     total_budget="$40-50",
#     meal_plans=[breakfast_meal, lunch_meal, dinner_meal],  # List of MealPlan objects
#     shopping_sections=[dairy_section, meat_section, pantry_section, produce_section],  # List of ShoppingCategory objects
#     shopping_tips=[...]
# )

Setting up LLM and essential tools

In [19]:
from crewai_tools import SerperDevTool
from crewai import Agent, Task, Crew, Process, LLM

In [20]:
llm = LLM(model="gemini/gemini-2.5-flash")

Create AI Agent worflow with CrewAI

In [21]:
meal_planner = Agent(
    role="Meal Planner & Recipe Researcher",
    goal="Search for optimal recipes and create detailed meal plans",
    backstory="A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

Define meal planning task

In [22]:
meal_planning_task = Task(
    description=(
        "Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. "
        "Find recipes that match the skill level and provide complete ingredient lists with quantities."
    ),
    expected_output="A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.",
    agent=meal_planner,
    output_pydantic=MealPlan,
    output_file="meals.json"
)

Create and Run meal plan crew

In [ ]:
from crewai import Crew, Process

meal_planner_crew = Crew(
    agents=[meal_planner],
    tasks=[meal_planning_task],
    process=Process.sequential,  # Ensures tasks are executed in order
    verbose=True
)

meal_planner_result = meal_planner_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],       
        "cooking_skill": "beginner"                
    }
)

print("✅ Single meal planning completed!")
print("📋 Single Meal Results:")
print(meal_planner_result)

Create shopping organization agent

In [25]:
shopping_organizer = Agent(
    role="Shopping Organizer", 
    goal="Organize grocery lists by store sections efficiently",
    backstory="An experienced shopper who knows how to organize lists for quick store trips and considers dietary restrictions.",
    tools=[],
    llm=llm,
    verbose=False
)

Define shopping organization task

In [26]:
shopping_task = Task(
    description=(
        "Organize the ingredients from the '{meal_name}' meal plan into a grocery shopping list. "
        "Group items by store sections and estimate quantities for {servings} people. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill: {cooking_skill}. "
        "Stay within budget: {budget}."
    ),
    expected_output="An organized shopping list grouped by store sections with quantities and prices.",
    agent=shopping_organizer,
    context=[meal_planning_task],
    output_pydantic=GroceryShoppingPlan,
    output_file="shopping_list.json"
)

Build 2 agents grocery crew

In [28]:
two_agent_grocery_crew = Crew(
    agents=[meal_planner, shopping_organizer],  # Both agents
    tasks=[meal_planning_task, shopping_task],   # Both tasks
    process=Process.sequential,
    verbose=True
)

# Run the complete crew (this will do BOTH meal planning AND shopping)
shopping_result = await two_agent_grocery_crew.kickoff_async(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],      
        "cooking_skill": "beginner"               
    }
)

# Print the shopping results
print("✅ Complete meal planning + shopping completed!")
print("📋 Shopping Results:")
print(shopping_result)

┌───────────────────────── 🚀 Crew Execution Started ─────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: d3b7e4dc-6c66-4e84-8375-06f50cc31781                                   │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────── 📋 Task Started ──────────────────────────────┐
│                                                                             │
│  Task Started                                                               │
│  Name: Search for the best 'Chicken Stir Fry' recipe for 4 people within a  │
│  $25 budget. Consider dietary restricti

┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: Meal Planner & Recipe Researcher                                    │
│                                                                             │
│  Task: Search for the best 'Chicken Stir Fry' recipe for 4 people within a  │
│  $25 budget. Consider dietary restrictions: ['no nuts'] and cooking skill   │
│  level: beginner. Find recipes that match the skill level and provide       │
│  complete ingredient lists with quantities.                                 │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘



ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


┌────────────────────── 🔧 Tool Execution Started (#1) ───────────────────────┐
│                                                                             │
│  Tool: search_the_internet_with_serper                                      │
│  Args: {'search_query': 'beginner chicken stir fry recipe no nuts for 4     │
│  people with ingredient list'}                                              │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌───────────────────── ✅ Tool Execution Completed (#1) ──────────────────────┐
│                                                                             │
│  Tool Completed                                                             │
│  Tool: search_the_internet_with_serper                                      │
│  Output: {'searchParameters': {'q': 'be

ERROR:crewai.telemetry.telemetry:('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Shopping Organizer                                                  │
│                                                                             │
│  Final Answer:                                                              │
│  total_budget='$25' meal_plans=[MealPlan(meal_name='Fast & Easy Chicken     │
│  Stir Fry with Vegetables (Nut-Free)', difficulty_level='Easy',             │
│  servings=4, researched_ingredients=['1.5 lbs (approx 680g) boneless,       │
│  skinless chicken breast, cut into bite-sized pieces', '6 tbsp low-sodium   │
│  soy sauce (2 tbsp for marinade, 4 tbsp for sauce)', '2 tbsp cornstarch (1  │
│  tbsp for marinade, 1 tbsp for sauce)', '1 tsp sugar (for marinade)', '1/2  │
│  cup chicken broth or water', '1 tbsp rice vinegar', '1 tbsp brown sugar    │
│  or honey', '1 clove garlic, minced', '

✅ Complete meal planning + shopping completed!
📋 Shopping Results:
total_budget='$25' meal_plans=[MealPlan(meal_name='Fast & Easy Chicken Stir Fry with Vegetables (Nut-Free)', difficulty_level='Easy', servings=4, researched_ingredients=['1.5 lbs (approx 680g) boneless, skinless chicken breast, cut into bite-sized pieces', '6 tbsp low-sodium soy sauce (2 tbsp for marinade, 4 tbsp for sauce)', '2 tbsp cornstarch (1 tbsp for marinade, 1 tbsp for sauce)', '1 tsp sugar (for marinade)', '1/2 cup chicken broth or water', '1 tbsp rice vinegar', '1 tbsp brown sugar or honey', '1 clove garlic, minced', '1 tsp grated fresh ginger', '1 tbsp canola or vegetable oil', '1 head broccoli, cut into florets', '2 medium carrots, sliced', '1 red bell pepper, sliced', '1 cup snow peas', 'Cooked rice, for serving'])] shopping_sections=[ShoppingCategory(section_name='Produce', items=[GroceryItem(name='Broccoli', quantity='1 head', estimated_price='$2.00-$3.00', category='Produce'), GroceryItem(name='Carrots',

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

┌────────────────────────────── Crew Completion ──────────────────────────────┐
│                                                                             │
│  Crew Execution Completed                                                   │
│  Name: crew                                                                 │
│  ID: d3b7e4dc-6c66-4e84-8375-06f50cc31781                                   │
│  Final Output: {"total_budget":"$25","meal_plans":[{"meal_name":"Fast &     │
│  Easy Chicken Stir Fry with Vegetables                                      │
│  (Nut-Free)","difficulty_level":"Easy","servings":4,"researched_ingredient  │
│  s":["1.5 lbs (approx 680g) boneless, skinless chicken breast, cut into     │
│  bite-sized pieces","6 tbsp low-sodium soy sauce (2 tbsp for marinade, 4    │
│  tbsp for sauce)","2 tbsp cornstarch (1 tbsp for marinade, 1 tbsp for       │
│  sauce)","1 tsp sugar (for marinade)","1/2 cup chicken broth or water","1   │
│  tbsp rice vinegar","1 tbsp brown suga

ERROR:crewai.telemetry.telemetry:('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Adding financial intelligent with budget advisor agent

In [29]:
budget_advisor = Agent(
    role="Budget Advisor",
    goal="Provide cost estimates and money-saving tips",
    backstory="A budget-conscious shopper who helps families save money on groceries while respecting dietary needs.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

Define budget analysis task

In [30]:
budget_task = Task(
    description=(
        "Analyze the shopping plan for '{meal_name}' serving {servings} people. "
        "Ensure total cost stays within {budget}. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide practical money-saving tips and alternative ingredients if needed to meet budget."
    ),
    expected_output="A complete shopping guide with detailed prices, budget analysis, and money-saving tips.",
    agent=budget_advisor,
    context=[meal_planning_task, shopping_task],
    output_file="shopping_guide.md"
)

Use yaml with crewai

Using CrewBase and decorator in Crewai

In [33]:
from leftover import LeftoversCrew

leftovers_cb = LeftoversCrew(llm=llm)
yaml_leftover_manager = leftovers_cb.leftover_manager()
yaml_leftover_task    = leftovers_cb.leftover_task()

Define summary agent and task

In [34]:
summary_agent = Agent(
    role="Report Compiler",
    goal="Compile comprehensive meal planning reports from all team outputs",
    backstory="A skilled coordinator who organizes information from multiple specialists into comprehensive, easy-to-follow reports.",
    tools=[],
    llm=llm,
    verbose=False
)

In [35]:
summary_task = Task(
    description=(
        "Compile a comprehensive meal planning report that includes:\n"
        "1. The complete recipe and cooking instructions from the meal planner\n"
        "2. The organized shopping list with prices from the shopping organizer\n"
        "3. The budget analysis and money-saving tips from the budget advisor\n"
        "4. The leftover management suggestions from the waste reduction specialist\n"
        "Format this as a complete, user-friendly meal planning guide."
    ),
    expected_output="A comprehensive meal planning guide that combines all team outputs into one cohesive report.",
    agent=summary_agent,
    context=[meal_planning_task, shopping_task, budget_task, yaml_leftover_task],
)

Assembling complete grocery planning team

In [36]:
complete_grocery_crew = Crew(
    agents=[
        meal_planner,           
        shopping_organizer,      
        budget_advisor,         
        yaml_leftover_manager,  # YAML-based leftover manager
        summary_agent           # New summary agent
    ],
    tasks=[
        meal_planning_task,     
        shopping_task,          
        budget_task,            
        yaml_leftover_task,    # YAML-based leftover task
        summary_task            # New summary task
    ],
    process=Process.sequential,
    verbose=True
)

Executing complete workflow 

In [39]:
# Run the complete crew
complete_result = await complete_grocery_crew.kickoff_async(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",
        "dietary_restrictions": ["no nuts", "low sodium"],
        "cooking_skill": "beginner"
    }
)

print("✅ Complete meal planning with summary completed!")
print("📋 Complete Results:")
print(complete_result)

┌───────────────────────── 🚀 Crew Execution Started ─────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 9b20954e-ece6-4e59-ae4b-f79c04187fb5                                   │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────── 📋 Task Started ──────────────────────────────┐
│                                                                             │
│  Task Started                                                               │
│  Name: Search for the best 'Chicken Stir Fry' recipe for 4 people within a  │
│  $25 budget. Consider dietary restricti

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


┌───────────────────── ✅ Tool Execution Completed (#10) ─────────────────────┐
│                                                                             │
│  Tool Completed                                                             │
│  Tool: search_the_internet_with_serper                                      │
│  Output: {'searchParameters': {'q': 'beginner chicken stir fry recipe no    │
│  nuts low sodium for 4 people under $25', 'type': 'search', 'num': 10,      │
│  'engine': 'google'}, 'organic': [{'title': 'CHICKEN STIR FRY | easy,       │
│  healthy 30-minute dinner recipe!', 'link':                                 │
│  'https://www.youtube.com/watch?v=h6EaCv-lLQY', 'snippet': 'A good chicken  │
│  stir fry ticks all the boxes for the ideal weeknight dinner! It delivers   │
│  in flavor, simplicity, and a healthy balance of ...', 'position': 1},      │
│  {'title': 'Quick and Easy 15 Minute Chicken Stir-Fry - Oh Sweet Basil',    │
│  'link':                               

ERROR:crewai.telemetry.telemetry:('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Shopping Organizer                                                  │
│                                                                             │
│  Final Answer:                                                              │
│  total_budget='$25' meal_plans=[MealPlan(meal_name='Easy Chicken Stir       │
│  Fry', difficulty_level='Easy', servings=4, researched_ingredients=['1 lb   │
│  boneless, skinless chicken breast or thighs, cut into bite-sized pieces',  │
│  '1 head broccoli, cut into florets', '2 carrots, thinly sliced or          │
│  julienned', '1 bell pepper (any color), sliced', '1 onion, thinly          │
│  sliced', '1 cup snow peas or snap peas', '1/2 cup low sodium chicken       │
│  broth', '1/4 cup low sodium soy sauce', '1 tablespoon cornstarch', '1      │
│  tablespoon honey or brown sugar', '1 t

...

┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Report Compiler                                                     │
│                                                                             │
│  Final Answer:                                                              │
│  # Comprehensive Meal Planning Guide: Easy Chicken Stir Fry                 │
│                                                                             │
│  Welcome to your comprehensive meal planning guide for **Easy Chicken Stir  │
│  Fry**! This guide combines your recipe, a detailed shopping list with      │
│  prices, budget analysis, money-saving tips, and clever ideas for using up  │
│  leftovers, all tailored to your low-sodium and nut-free dietary needs.     │
│                                                                             │
│  ---                                   

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Complete meal planning with summary completed!
📋 Complete Results:
# Comprehensive Meal Planning Guide: Easy Chicken Stir Fry

Welcome to your comprehensive meal planning guide for **Easy Chicken Stir Fry**! This guide combines your recipe, a detailed shopping list with prices, budget analysis, money-saving tips, and clever ideas for using up leftovers, all tailored to your low-sodium and nut-free dietary needs.

---

## Meal Overview

*   **Meal Name:** Easy Chicken Stir Fry
*   **Difficulty Level:** Easy
*   **Servings:** 4
*   **Total Budget:** $25.00

---

## Part 1: The Easy Chicken Stir Fry Recipe

A quick, healthy, and flavorful dish perfect for a weeknight meal!

### Ingredients:

*   1 lb boneless, skinless chicken breast or thighs, cut into bite-sized pieces
*   1 head broccoli, cut into florets
*   2 carrots, thinly sliced or julienned
*   1 bell pepper (any color), sliced
*   1 onion, thinly sliced
*   1 cup snow peas or snap peas
*   2 tablespoons vegetable oil (or other

┌────────────────────────────── Crew Completion ──────────────────────────────┐
│                                                                             │
│  Crew Execution Completed                                                   │
│  Name: crew                                                                 │
│  ID: 9b20954e-ece6-4e59-ae4b-f79c04187fb5                                   │
│  Final Output: # Comprehensive Meal Planning Guide: Easy Chicken Stir Fry   │
│                                                                             │
│  Welcome to your comprehensive meal planning guide for **Easy Chicken Stir  │
│  Fry**! This guide combines your recipe, a detailed shopping list with      │
│  prices, budget analysis, money-saving tips, and clever ideas for using up  │
│  leftovers, all tailored to your low-sodium and nut-free dietary needs.     │
│                                                                             │
│  ---                                  

ERROR:crewai.telemetry.telemetry:('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Create Specialized dietary agent

In [42]:
# Your code here
nutrition_analyst = Agent(
    role="Nutrition Analyst & Health Advisor",  # TODO: Define the role
    goal="Analyze meal nutritional content and provide healthy recommendations",  # TODO: Define the goal  
    backstory="A certified nutritionist who evaluates meals for nutritional balance, calorie content," \
    "and health optiomization while respecting diertary restrictions.",  # TODO: Create backstory
    tools=[SerperDevTool()],  # TODO: Add appropriate tools
    llm=llm,
    verbose=False
)

# TODO: Create nutrition analysis task
nutrition_task = Task(
    description=(
        "Analyze the nutritional content of the '{meal_name}' meal plan for {servings} people. "
        "Calculate approximate calories, protein, carbs, and fats. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide healthy alternatives if the meal could be more nutritious while staying within {budget}."
    ),
    expected_output="Nutritional analysis with calorie estimates, macronutrient breakdown, and healthy improvement suggestions.",
    agent=nutrition_analyst,
    context=[meal_planning_task, shopping_task, budget_task],
    output_file="nutrition_analysis.md"
)
# TODO: Add to crew and test
health_focused_crew = Crew(
    agents=[meal_planner, shopping_organizer, budget_advisor, nutrition_analyst, yaml_leftover_manager, summary_agent],
    tasks=[meal_planning_task, shopping_task, budget_task, nutrition_task, yaml_leftover_task, summary_task],
    process=Process.sequential,
    verbose=True
)

result = await health_focused_crew.kickoff_async(
    inputs={
        "meal_name": "Quinoa Buddha Bowl",
        "servings": 2,
        "budget": "$20",
        "dietary_restrictions": ["vegetarian", "high protein"],
        "cooking_skill": "intermediate"
    }
)

┌───────────────────────── 🚀 Crew Execution Started ─────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 17ae8e04-e2a1-49f0-8401-e94979b0e8c2                                   │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────── 📋 Task Started ──────────────────────────────┐
│                                                                             │
│  Task Started                                                               │
│  Name: Search for the best 'Quinoa Buddha Bowl' recipe for 2 people within  │
│  a $20 budget. Consider dietary restric

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Meal Planner & Recipe Researcher                                    │
│                                                                             │
│  Final Answer:                                                              │
│  {"meal_name": "High-Protein Vegetarian Quinoa Buddha Bowl",                │
│  "difficulty_level": "Medium", "servings": 2, "researched_ingredients":     │
│  ["1 cup uncooked quinoa", "8 oz firm or extra-firm tofu, pressed and       │
│  cubed", "1 can (15 oz) chickpeas, rinsed and drained", "1 large avocado,   │
│  sliced", "2 cups mixed greens (e.g., spinach, kale, or spring mix)", "1    │
│  large sweet potato, peeled and cubed", "1 cup broccoli florets", "1/2 red  │
│  onion, thinly sliced", "1/2 cup cherry tomatoes, halved", "3 tablespoons   │
│  tahini", "2 tablespoons lemon juice", 

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Shopping Organizer                                                  │
│                                                                             │
│  Final Answer:                                                              │
│  total_budget='$20' meal_plans=[MealPlan(meal_name='High-Protein            │
│  Vegetarian Quinoa Buddha Bowl', difficulty_level='Medium', servings=2,     │
│  researched_ingredients=['1 cup uncooked quinoa', '8 oz firm or extra-firm  │
│  tofu, pressed and cubed', '1 can (15 oz) chickpeas, rinsed and drained',   │
│  '1 large avocado, sliced', '2 cups mixed greens (e.g., spinach, kale, or   │
│  spring mix)', '1 large sweet potato, peeled and cubed', '1 cup broccoli    │
│  florets', '1/2 red onion, thinly sliced', '1/2 cup cherry tomatoes,        │
│  halved', '3 tablespoons tahini', '2 ta

...

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


┌─────────────────────────── ✅ Agent Final Answer ───────────────────────────┐
│                                                                             │
│  Agent: Report Compiler                                                     │
│                                                                             │
│  Final Answer:                                                              │
│  # Your Comprehensive Meal Planning Guide: High-Protein Vegetarian Quinoa   │
│  Buddha Bowl                                                                │
│                                                                             │
│  Welcome to your comprehensive meal planning guide! This report combines a  │
│  delicious, high-protein vegetarian recipe with a smart shopping list,      │
│  detailed budget analysis, and clever tips to minimize food waste. Get      │
│  ready to enjoy a nutritious meal and make the most of your ingredients!    │
│                                        

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

┌────────────────────────────── Crew Completion ──────────────────────────────┐
│                                                                             │
│  Crew Execution Completed                                                   │
│  Name: crew                                                                 │
│  ID: 17ae8e04-e2a1-49f0-8401-e94979b0e8c2                                   │
│  Final Output: # Your Comprehensive Meal Planning Guide: High-Protein       │
│  Vegetarian Quinoa Buddha Bowl                                              │
│                                                                             │
│  Welcome to your comprehensive meal planning guide! This report combines a  │
│  delicious, high-protein vegetarian recipe with a smart shopping list,      │
│  detailed budget analysis, and clever tips to minimize food waste. Get      │
│  ready to enjoy a nutritious meal and make the most of your ingredients!    │
│                                       

ERROR:crewai.telemetry.telemetry:('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


In [43]:
from enum import Enum
from typing import Dict

class MealType(str, Enum):
    BREAKFAST = "breakfast"
    LUNCH = "lunch" 
    DINNER = "dinner"
    SNACK = "snack"

class DailyMeals(BaseModel):
    """Meals for one day"""
    date: str = Field(description="Date in YYYY-MM-DD format")
    breakfast: Optional[MealPlan] = Field(default=None, description="Breakfast meal plan")
    lunch: Optional[MealPlan] = Field(default=None, description="Lunch meal plan") 
    dinner: Optional[MealPlan] = Field(default=None, description="Dinner meal plan")
    snacks: Optional[List[MealPlan]] = Field(default=None, description="Snack meal plans")
class WeeklyMealPlan(BaseModel):
    """Complete weekly meal planning"""
    week_start_date: str = Field(description="Start date of the week")
    daily_meals: List[DailyMeals] = Field(description="Meals for each day")
    weekly_themes: List[str] = Field(description="Cooking themes for the week")
    prep_suggestions: List[str] = Field(description="Meal prep recommendations")

class WeeklyGroceryPlan(BaseModel):
    """Weekly grocery shopping strategy"""
    weekly_budget: str = Field(description="Total weekly budget")
    meal_plans: List[DailyMeals] = Field(description="All weekly meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections")
    bulk_items: List[GroceryItem] = Field(description="Items to buy in bulk")
    shopping_tips: List[str] = Field(description="Weekly shopping efficiency tips")
    budget_breakdown: Dict[str, str] = Field(description="Daily budget allocation")

# Test the models
sample_weekly_plan = WeeklyMealPlan(
    week_start_date="2024-01-15",
    daily_meals=[
        DailyMeals(
            date="2024-01-15",
            breakfast=MealPlan(meal_name="Oatmeal", difficulty_level="Easy", servings=2, researched_ingredients=["oats", "milk", "berries"]),
            lunch=MealPlan(meal_name="Salad", difficulty_level="Easy", servings=2, researched_ingredients=["lettuce", "tomatoes", "dressing"]),
            dinner=MealPlan(meal_name="Pasta", difficulty_level="Medium", servings=2, researched_ingredients=["pasta", "sauce", "cheese"])
        )
    ],
    weekly_themes=["Italian Monday", "Taco Tuesday"],
    prep_suggestions=["Wash vegetables on Sunday", "Cook grains in bulk"]
)

display(JSON(sample_weekly_plan.model_dump()))

<IPython.core.display.JSON object>